In [0]:
# Install pytest
%pip install pytest

print("Pytest installed successfully")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Pytest installed successfully


In [0]:
# Imports
import pytest
from pyspark.sql.functions import col
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

print("Imports successful")

Imports successful


In [0]:
# Define deduplication function to test
def deduplicate(df, merge_key, watermark_col):
    
    window = Window \
        .partitionBy(merge_key) \
        .orderBy(col(watermark_col).desc())
    
    deduped_df = df \
        .withColumn("rn", row_number().over(window)) \
        .filter("rn = 1") \
        .drop("rn")
    
    return deduped_df

# Test 1 - Deduplication removes duplicates
def test_deduplication_removes_duplicates():
    
    # Create fake input data with duplicates
    input_data = [
        ("CLM001", "PENDING",  "2024-01-10"),
        ("CLM001", "APPROVED", "2024-01-15"),  # duplicate
        ("CLM002", "DENIED",   "2024-01-12"),
    ]
    
    input_df = spark.createDataFrame(
        input_data,
        ["claim_id", "claim_status", "updated_at"]
    ).withColumn("updated_at", col("updated_at").cast("timestamp"))
    
    # Run deduplication
    result_df = deduplicate(input_df, "claim_id", "updated_at")
    
    # Check results
    assert result_df.count() == 2, \
        f"Expected 2 rows after dedup but got {result_df.count()}"
    
    clm001 = result_df \
        .filter("claim_id = 'CLM001'") \
        .collect()[0]
    
    assert clm001["claim_status"] == "APPROVED", \
        f"Expected APPROVED but got {clm001['claim_status']}"
    
    print("✅ test_deduplication_removes_duplicates PASSED")

# Run the test
test_deduplication_removes_duplicates()

✅ test_deduplication_removes_duplicates PASSED


In [0]:
# Test 2 - Watermark filter only returns new records
def test_watermark_filters_old_records():
    
    # Create fake input data
    input_data = [
        ("CLM001", "2024-01-10"),  # older than watermark
        ("CLM002", "2024-01-16"),  # newer than watermark
        ("CLM003", "2024-01-17"),  # newer than watermark
    ]
    
    input_df = spark.createDataFrame(
        input_data,
        ["claim_id", "updated_at"]
    ).withColumn("updated_at", col("updated_at").cast("timestamp"))
    
    # Apply watermark filter
    watermark = "2024-01-15"
    result_df = input_df.filter(f"updated_at > '{watermark}'")
    
    # Check results
    assert result_df.count() == 2, \
        f"Expected 2 rows after watermark filter but got {result_df.count()}"
    
    claim_ids = [row["claim_id"] for row in result_df.collect()]
    
    assert "CLM001" not in claim_ids, \
        "CLM001 should have been filtered out — it is older than watermark"
    
    assert "CLM002" in claim_ids, \
        "CLM002 should be included — it is newer than watermark"
    
    assert "CLM003" in claim_ids, \
        "CLM003 should be included — it is newer than watermark"
    
    print("✅ test_watermark_filters_old_records PASSED")

# Run the test
test_watermark_filters_old_records()

✅ test_watermark_filters_old_records PASSED


In [0]:
# Test 3 - Data quality catches bad data
def test_data_quality_catches_bad_data():
    
    # Create bad data
    bad_data = [
        ("CLM001", "PAT101", "PRV01", "APPROVED",  1500.00),  # good
        ("CLM002", None,     "PRV02", "PENDING",    800.00),   # null patient_id
        ("CLM003", "PAT103", "PRV03", "UNKNOWN",    2200.00),  # invalid status
        ("CLM004", "PAT104", "PRV01", "APPROVED",  -500.00),   # negative amount
    ]
    
    bad_df = spark.createDataFrame(
        bad_data,
        ["claim_id", "patient_id", "provider_id", "claim_status", "claim_amount"]
    ).toPandas()
    
    # Run validation rules
    valid_statuses = ["PENDING", "APPROVED", "DENIED", "PAID"]
    
    rule1 = bad_df["claim_id"].isnull().sum() == 0
    rule2 = (bad_df["claim_amount"] > 0).all()
    rule3 = bad_df["claim_status"].isin(valid_statuses).all()
    rule4 = bad_df["patient_id"].isnull().sum() == 0
    
    # Check results
    assert rule1 == True, \
        "claim_id not null check should pass"
    
    assert rule2 == False, \
        "claim_amount positive check should fail — has negative amount"
    
    assert rule3 == False, \
        "claim_status valid values check should fail — has UNKNOWN"
    
    assert rule4 == False, \
        "patient_id not null check should fail — has null"
    
    print("✅ test_data_quality_catches_bad_data PASSED")

# Run the test
test_data_quality_catches_bad_data()

✅ test_data_quality_catches_bad_data PASSED


In [0]:
# Test 4 - Good data passes validation
def test_good_data_passes_validation():
    
    # Create good data
    good_data = [
        ("CLM001", "PAT101", "PRV01", "APPROVED", 1500.00),
        ("CLM002", "PAT102", "PRV02", "PENDING",   800.00),
        ("CLM003", "PAT103", "PRV03", "DENIED",   2200.00),
        ("CLM004", "PAT104", "PRV01", "PAID",     3100.00),
    ]
    
    good_df = spark.createDataFrame(
        good_data,
        ["claim_id", "patient_id", "provider_id", "claim_status", "claim_amount"]
    ).toPandas()
    
    # Run validation rules
    valid_statuses = ["PENDING", "APPROVED", "DENIED", "PAID"]
    
    rule1 = good_df["claim_id"].isnull().sum() == 0
    rule2 = (good_df["claim_amount"] > 0).all()
    rule3 = good_df["claim_status"].isin(valid_statuses).all()
    rule4 = good_df["patient_id"].isnull().sum() == 0
    rule5 = good_df["provider_id"].isnull().sum() == 0
    
    # All rules should pass
    assert rule1 == True, "claim_id not null should pass"
    assert rule2 == True, "claim_amount positive should pass"
    assert rule3 == True, "claim_status valid values should pass"
    assert rule4 == True, "patient_id not null should pass"
    assert rule5 == True, "provider_id not null should pass"
    
    print("✅ test_good_data_passes_validation PASSED")

# Run the test
test_good_data_passes_validation()

✅ test_good_data_passes_validation PASSED


In [0]:
# Run all tests together
print("=== RUNNING ALL PYTEST TESTS ===")
print("=" * 50)

tests = [
    test_deduplication_removes_duplicates,
    test_watermark_filters_old_records,
    test_data_quality_catches_bad_data,
    test_good_data_passes_validation
]

passed = 0
failed = 0

for test in tests:
    try:
        test()
        passed += 1
    except AssertionError as e:
        print(f"❌ {test.__name__} FAILED: {e}")
        failed += 1

print("=" * 50)
print(f"Total tests: {len(tests)}")
print(f"Passed:      {passed}")
print(f"Failed:      {failed}")

if failed == 0:
    print("ALL TESTS PASSED ✅ — Safe to deploy")
else:
    print("SOME TESTS FAILED ❌ — Fix before deploying")

=== RUNNING ALL PYTEST TESTS ===
✅ test_deduplication_removes_duplicates PASSED
✅ test_watermark_filters_old_records PASSED
✅ test_data_quality_catches_bad_data PASSED
✅ test_good_data_passes_validation PASSED
Total tests: 4
Passed:      4
Failed:      0
ALL TESTS PASSED ✅ — Safe to deploy
